In [35]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import plotly.express as px

# === INPUT PATHS ===
input_folder = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed/"
file_name_list_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/file_name_list.csv"
common_mzs_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters.csv"
image_output_folder = "/home/ajarrah/PhD_Thesis/chapter_2/results/vmax/images_individual"
results_output_folder = "/home/ajarrah/PhD_Thesis/chapter_2/results/vmax/"

In [36]:
# Read file names from CSV
file_name_df = pd.read_csv(file_name_list_csv_input_path)
file_names = file_name_df["file_name"].tolist()

In [37]:

# Create mapping from file name → short name
file_name_dict = {}
for fn in file_names:
    parts = fn.split("_")
    group = parts[0][0]         # First letter of AFM/YFM
    condition = parts[1]        # AD or C
    sample_num = parts[2]       # Sample number
    short_name = f"{group}{condition}_{sample_num}"
    file_name_dict[fn] = short_name

In [38]:
# Your list of sample IDs (also used as keys in the dictionary)
aad_1 = sc.read_h5ad(os.path.join(input_folder, "AFM_AD_1_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
aad_2 = sc.read_h5ad(os.path.join(input_folder, "AFM_AD_2_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
aad_3 = sc.read_h5ad(os.path.join(input_folder, "AFM_AD_3_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
aad_4 = sc.read_h5ad(os.path.join(input_folder, "AFM_AD_4_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
ac_1 = sc.read_h5ad(os.path.join(input_folder, "AFM_C_1_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
ac_2 = sc.read_h5ad(os.path.join(input_folder, "AFM_C_2_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
ac_3 = sc.read_h5ad(os.path.join(input_folder, "AFM_C_3_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
ac_4 = sc.read_h5ad(os.path.join(input_folder, "AFM_C_4_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
yad_1 = sc.read_h5ad(os.path.join(input_folder, "YFM_AD_1_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
yad_2 = sc.read_h5ad(os.path.join(input_folder, "YFM_AD_2_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
yad_3 = sc.read_h5ad(os.path.join(input_folder, "YFM_AD_3_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
yad_4 = sc.read_h5ad(os.path.join(input_folder, "YFM_AD_4_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
yc_1 = sc.read_h5ad(os.path.join(input_folder, "YFM_C_1_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
yc_2 = sc.read_h5ad(os.path.join(input_folder, "YFM_C_2_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
yc_3 = sc.read_h5ad(os.path.join(input_folder, "YFM_C_3_peaks_0.00001_top1000_9w_3p_5points.h5ad"))
yc_4 = sc.read_h5ad(os.path.join(input_folder, "YFM_C_4_peaks_0.00001_top1000_9w_3p_5points.h5ad"))


In [39]:
sample_list = [
    aad_1, aad_2, aad_3, aad_4,
    ac_1, ac_2, ac_3, ac_4,
    yad_1, yad_2, yad_3, yad_4,
    yc_1, yc_2, yc_3, yc_4
]

sample_ids = [
    "aad_1", "aad_2", "aad_3", "aad_4",
    "ac_1", "ac_2", "ac_3", "ac_4",
    "yad_1", "yad_2", "yad_3", "yad_4",
    "yc_1", "yc_2", "yc_3", "yc_4"
]

# Map sample ID → AnnData object
sample_map = dict(zip(sample_ids, sample_list))


In [40]:
# Assuming sample_list contains your AnnData objects and their variable names are like 'aad_1', 'ac_1', etc.
# They must be defined variables or in a dictionary for easy iteration.

# Example: if you have them in a dictionary like sample_map = {'aad_1': aad_1, ...}

for sample_id in sample_list:
    adata = sample_id  # Get the AnnData object by variable name string
    # Sum all intensities for each spot (row) across all m/z (columns)
    tic = adata.X.sum(axis=1)  # shape (n_spots, 1) or (n_spots,) depending on sparse or dense
    
    # If sparse, convert to flat array
    if hasattr(tic, "A1"):  # sparse matrix
        tic = tic.A1
    else:
        tic = np.asarray(tic).flatten()

    # Add TIC as a new column in obs
    adata.obs["Processed_TIC"] = tic

In [41]:
# === STEP 3: Load common m/z list ===
common_mz_df = pd.read_csv(common_mzs_csv_input_path)
common_mz_df

,mzs,group,sample,common_group_name
0,326.1952,AC,1,326.1952
1,326.1952,AAD,1,326.1952
2,326.1952,AAD,3,326.1952
3,326.1952,AAD,2,326.1952
4,326.1952,AAD,4,326.1952
...,...,...,...,...
8555,1256.9180,AAD,4,1256.9080
8556,1256.9180,AC,1,1256.9080
8557,1256.9218,AC,3,1256.9080
8558,1256.9218,AC,4,1256.9080


In [42]:

# Example: convert group/sample to sample_id that matches your sample_map keys
def make_sample_id(row):
    # lower group, then underscore, then sample number
    return f"{row['group'].lower()}_{row['sample']}"

common_mz_df['sample_id'] = common_mz_df.apply(make_sample_id, axis=1)
print(common_mz_df)

            mzs group  sample  common_group_name sample_id
0      326.1952    AC       1           326.1952      ac_1
1      326.1952   AAD       1           326.1952     aad_1
2      326.1952   AAD       3           326.1952     aad_3
3      326.1952   AAD       2           326.1952     aad_2
4      326.1952   AAD       4           326.1952     aad_4
...         ...   ...     ...                ...       ...
8555  1256.9180   AAD       4          1256.9080     aad_4
8556  1256.9180    AC       1          1256.9080      ac_1
8557  1256.9218    AC       3          1256.9080      ac_3
8558  1256.9218    AC       4          1256.9080      ac_4
8559  1256.9218    AC       2          1256.9080      ac_2

[8560 rows x 5 columns]


In [43]:
pivot_df = common_mz_df.pivot(index='sample_id', columns='common_group_name', values='mzs')
pivot_df

common_group_name,326.1952,337.1898,339.2275,340.2314,341.2417,351.1685,351.2282,352.2324,353.2430,354.2494,...,1023.6721,1024.6758,1025.6855,1088.8147,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9080
sample_id,,,,,,,,,,,,,,,,,,,,,
aad_1,326.1952,337.1898,339.2275,340.2314,341.2434,351.1685,351.2282,352.2324,353.2430,354.2512,...,1023.6736,1024.6824,1025.6922,1088.8234,1114.8364,1115.8402,1140.7520,1142.7729,1255.9191,1256.9180
aad_2,326.1952,337.1898,339.2275,340.2314,341.2434,351.1685,351.2282,352.2324,353.2430,354.2512,...,1023.6736,1024.6824,1025.6922,1088.8234,1114.8364,1115.8402,1140.7520,1142.7729,1255.9191,1256.9180
aad_3,326.1952,337.1898,339.2275,340.2314,341.2434,351.1685,351.2282,352.2324,353.2430,354.2494,...,1023.6736,1024.6773,1025.6871,1088.8180,1114.8364,1115.8402,1140.7520,1142.7729,1255.9128,1256.9180
aad_4,326.1952,337.1898,339.2275,340.2314,341.2434,351.1685,351.2282,352.2324,353.2430,354.2494,...,1023.6736,1024.6824,1025.6922,1088.8180,1114.8364,1115.8402,1140.7520,1142.7729,1255.9128,1256.9180
ac_1,326.1952,337.1898,339.2275,340.2314,341.2417,351.1685,351.2282,352.2324,353.2430,354.2512,...,1023.6787,1024.6824,1025.6922,1088.8180,1114.8364,1115.8402,1140.7577,1142.7729,1255.9128,1256.9180
ac_2,326.1990,337.1936,339.2313,340.2352,341.2455,351.1723,351.2303,352.2362,353.2468,354.2550,...,1023.6825,1024.6862,1025.6909,1088.8218,1114.8402,1115.8440,1140.7615,1142.7710,1255.9166,1256.9218
ac_3,326.1990,337.1919,339.2313,340.2335,341.2455,351.1723,351.2303,352.2362,353.2468,354.2532,...,1023.6774,1024.6811,1025.6909,1088.8218,1114.8402,1115.8440,1140.7615,1142.7710,1255.9166,1256.9218
ac_4,326.1990,337.1919,339.2313,340.2335,341.2455,351.1723,351.2303,352.2362,353.2468,354.2532,...,1023.6774,1024.6811,1025.6909,1088.8218,1114.8402,1115.8440,1140.7615,1142.7710,1255.9166,1256.9218
yad_1,326.2060,337.1987,339.2380,340.2402,341.2522,351.1787,351.2367,352.2426,353.2532,354.2596,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118


In [44]:
order = [
    'yc_1', 'yc_2', 'yc_3', 'yc_4',
    'yad_1', 'yad_2', 'yad_3', 'yad_4',
    'ac_1', 'ac_2', 'ac_3', 'ac_4',
    'aad_1', 'aad_2', 'aad_3', 'aad_4'
]

# Assuming pivot_df is your pivoted DataFrame with index = sample_id
pivot_df_sorted = pivot_df.loc[pivot_df.index.intersection(order)]
pivot_df_sorted = pivot_df_sorted.loc[order]

pivot_df_sorted

common_group_name,326.1952,337.1898,339.2275,340.2314,341.2417,351.1685,351.2282,352.2324,353.2430,354.2494,...,1023.6721,1024.6758,1025.6855,1088.8147,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9080
sample_id,,,,,,,,,,,,,,,,,,,,,
yc_1,326.2076,337.2004,339.2380,340.2419,341.2539,351.1787,351.2384,352.2426,353.2532,354.2613,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8331,1115.8369,1140.7539,1142.7633,1255.9067,1256.9118
yc_2,326.2038,337.1966,339.2359,340.2381,341.2501,351.1767,351.2346,352.2406,353.2511,354.2575,...,1023.6734,1024.6771,1025.6869,1088.8169,1114.8293,1115.8331,1140.7501,1142.7653,1255.9092,1256.9080
yc_3,326.2055,337.1983,339.2359,340.2398,341.2518,351.1767,351.2364,352.2406,353.2511,354.2593,...,1023.6734,1024.6771,1025.6869,1088.8169,1114.8349,1115.8387,1140.7501,1142.7653,1255.9092,1256.9143
yc_4,326.1995,337.1924,339.2317,340.2339,341.2459,351.1726,351.2305,352.2365,353.2471,354.2535,...,1023.6760,1024.6797,1025.6895,1088.8147,1114.8329,1115.8367,1140.7539,1142.7691,1255.9141,1256.9130
yad_1,326.2060,337.1987,339.2380,340.2402,341.2522,351.1787,351.2367,352.2426,353.2532,354.2596,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118
yad_2,326.2060,337.2004,339.2380,340.2402,341.2522,351.1787,351.2367,352.2426,353.2532,354.2596,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118
yad_3,326.2060,337.2004,339.2380,340.2419,341.2522,351.1787,351.2384,352.2426,353.2532,354.2613,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118
yad_4,326.2060,337.1987,339.2380,340.2402,341.2522,351.1787,351.2367,352.2426,353.2532,354.2596,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118
ac_1,326.1952,337.1898,339.2275,340.2314,341.2417,351.1685,351.2282,352.2324,353.2430,354.2512,...,1023.6787,1024.6824,1025.6922,1088.8180,1114.8364,1115.8402,1140.7577,1142.7729,1255.9128,1256.9180


In [45]:
from matplotlib.colors import LinearSegmentedColormap
# === COLOR SCALE ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

In [46]:
# Create output folder if it doesn't exist
os.makedirs(image_output_folder, exist_ok=True)

In [47]:
pivot_df_sorted.columns

Index([ 326.1952,  337.1898,  339.2275,  340.2314,  341.2417,  351.1685,
        351.2282,  352.2324,   353.243,  354.2494,
       ...
       1023.6721, 1024.6758, 1025.6855, 1088.8147, 1114.8276, 1115.8313,
       1140.7482, 1142.7633, 1255.9067,  1256.908],
      dtype='float64', name='common_group_name', length=535)

In [48]:
import matplotlib.pyplot as plt
import numpy as np
import os

def extract_spatial_mz_data(adata, mz_val, tol=0.01):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        return None, None, None
    mz_index = np.argmin(mz_diff)
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    return x, y, intensities

# Iterate over columns (common_group_name)
for common_group in pivot_df_sorted.columns:
    mz_val = float(common_group)  # convert column name to float if needed

    # Prepare subplots: 4x4 grid (adjust if you have a different number of samples)
    n_samples = len(pivot_df_sorted.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))

    fig, axs = plt.subplots(rows, cols, figsize=(cols*4, rows*4), squeeze=False)
    axs = axs.flatten()

    for i, sample_id in enumerate(pivot_df_sorted.index):
        adata = sample_map.get(sample_id)

        val = pivot_df_sorted.loc[sample_id, mz_val]
        x, y, intensities = extract_spatial_mz_data(adata, val, tol=0.001)

        # Compute vmax as the 97th percentile (top 3% saturated)
        vmax_value = np.percentile(intensities, 99.9)

        ax = axs[i]
        sc = ax.scatter(x, y, c=intensities, cmap=custom_cmap, marker='s', s=8,
                        vmin=0, vmax=vmax_value)
        ax.set_title(f"{sample_id} m/z={val:.4f}")
        ax.axis('off')

        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

    # Turn off any unused subplots
    for j in range(i+1, len(axs)):
        axs[j].axis('off')

    fig.suptitle(f"Common Group: {common_group}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Save figure
    fig.savefig(os.path.join(image_output_folder, f"{common_group}.png"))
    plt.close(fig)


In [49]:
os.makedirs(image_output_folder + '_tic', exist_ok=True)

def extract_spatial_mz_data_tic(adata, mz_val, tol=0.01):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        return None, None, None
    mz_index = np.argmin(mz_diff)
    
    # Extract raw intensities for the mz_val
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    
    # Extract spatial coordinates
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    
    # Extract TIC for each spot (assuming column name is 'Processed_TIC')
    tic = adata.obs["Processed_TIC"].values
    
    # Normalize intensities by TIC, avoid division by zero
    normalized_intensities = intensities / tic
    normalized_intensities = np.nan_to_num(normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0)
    
    return x, y, normalized_intensities

# Iterate over columns (common_group_name)
for common_group in pivot_df_sorted.columns:
    mz_val = float(common_group)  # convert column name to float if needed

    # Prepare subplots: 4x4 grid (adjust if you have a different number of samples)
    n_samples = len(pivot_df_sorted.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4), squeeze=False)
    axs = axs.flatten()

    for i, sample_id in enumerate(pivot_df_sorted.index):
        adata = sample_map.get(sample_id)
        
        val = pivot_df_sorted.loc[sample_id, mz_val]
        x, y, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)

        # Compute vmax as 97th percentile (top 3% saturates)
        vmax_value = np.percentile(intensities, 99.9)

        ax = axs[i]
        sc = ax.scatter(x, y, c=intensities, cmap=custom_cmap, marker='s', s=8,
                        vmin=0, vmax=vmax_value)
        ax.set_title(f"{sample_id} m/z={val:.4f}")
        ax.axis('off')

        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

    # Turn off any unused subplots
    for j in range(i + 1, len(axs)):
        axs[j].axis('off')

    fig.suptitle(f"Common Group: {common_group} TIC Normalized", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Save figure
    fig.savefig(os.path.join(image_output_folder + '_tic/', f"{common_group}.png"))
    plt.close(fig)


In [50]:
os.makedirs(results_output_folder + 'images_row_tic', exist_ok=True)

def extract_spatial_mz_data_tic(adata, mz_val, tol=0.01):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        return None, None, None
    mz_index = np.argmin(mz_diff)
    
    # Extract raw intensities for the mz_val
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    
    # Extract spatial coordinates
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    
    # Extract TIC for each spot (assuming column name is 'Processed_TIC')
    tic = adata.obs["Processed_TIC"].values
    
    # Normalize intensities by TIC, avoid division by zero
    normalized_intensities = intensities / tic
    normalized_intensities = np.nan_to_num(normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0)
    
    return x, y, normalized_intensities

for common_group in pivot_df_sorted.columns:
    mz_val = float(common_group)

    n_samples = len(pivot_df_sorted.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4), squeeze=False)
    axs = axs.flatten()

    for row_idx in range(rows):
        start_idx = row_idx * cols
        end_idx = min(start_idx + cols, n_samples)
        row_sample_ids = pivot_df_sorted.index[start_idx:end_idx]

        # --- Compute vmax for this row using 97th percentile ---
        all_intensities = []
        for sample_id in row_sample_ids:
            adata = sample_map.get(sample_id)
            val = pivot_df_sorted.loc[sample_id, mz_val]
            _, _, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)
            all_intensities.extend(intensities)
        vmax = np.percentile(all_intensities, 99.9)  # top 3% clipped
        vmin = 0

        # --- Plot each sample in this row ---
        for col_idx, sample_id in enumerate(row_sample_ids):
            i = start_idx + col_idx
            adata = sample_map.get(sample_id)
            val = pivot_df_sorted.loc[sample_id, mz_val]
            x, y, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)

            ax = axs[i]
            sc = ax.scatter(x, y, c=intensities, cmap=custom_cmap, marker='s', s=8,
                            vmin=vmin, vmax=vmax)
            ax.set_title(f"{sample_id} m/z={val:.4f}")
            ax.axis('off')

        # Add colorbar for the row (last axis in row)
        last_ax_in_row = axs[start_idx + len(row_sample_ids) - 1]
        fig.colorbar(sc, ax=last_ax_in_row, fraction=0.046, pad=0.04)

    # Turn off unused subplots
    for j in range(n_samples, rows * cols):
        axs[j].axis('off')

    fig.suptitle(f"Common Group: {common_group} TIC Normalized", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(os.path.join(results_output_folder + 'images_row_tic/', f"{common_group}.png"))
    plt.close(fig)

In [51]:
os.makedirs(results_output_folder + 'images_all_tic', exist_ok=True)

def extract_spatial_mz_data_tic(adata, mz_val, tol=0.01):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        return None, None, None
    mz_index = np.argmin(mz_diff)
    
    # Extract raw intensities for the mz_val
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    
    # Extract spatial coordinates
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    
    # Extract TIC for each spot (assuming column name is 'TIC')
    tic = adata.obs["Processed_TIC"].values
    
    # Normalize intensities by TIC, avoid division by zero
    normalized_intensities = intensities / tic
    normalized_intensities = np.nan_to_num(normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0)
    
    return x, y, normalized_intensities

for common_group in pivot_df_sorted.columns:
    mz_val = float(common_group)

    n_samples = len(pivot_df_sorted.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4), squeeze=False)
    axs = axs.flatten()

    # --- Compute global vmax using 97th percentile (clipping top 3%) ---
    all_intensities = []
    for sample_id in pivot_df_sorted.index:
        adata = sample_map.get(sample_id)
        val = pivot_df_sorted.loc[sample_id, mz_val]
        _, _, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)
        all_intensities.extend(intensities)

    vmax = np.percentile(all_intensities, 99.9)  # 97th percentile
    vmin = 0

    # --- Plot all samples ---
    for i, sample_id in enumerate(pivot_df_sorted.index):
        adata = sample_map.get(sample_id)
        val = pivot_df_sorted.loc[sample_id, mz_val]
        x, y, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)

        ax = axs[i]
        sc = ax.scatter(x, y, c=intensities, cmap=custom_cmap, marker='s', s=8,
                        vmin=vmin, vmax=vmax)
        ax.set_title(f"{sample_id} m/z={val:.4f}")
        ax.axis('off')

    # Turn off unused subplots
    for j in range(n_samples, rows * cols):
        axs[j].axis('off')

    # Place ONE colorbar outside the grid at far right
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # [left, bottom, width, height]
    fig.colorbar(sc, cax=cbar_ax)

    fig.suptitle(f"Common Group: {common_group} TIC Normalized", fontsize=16)
    plt.tight_layout(rect=[0, 0, 0.9, 0.96])  # leave space on right for cbar

    fig.savefig(os.path.join(results_output_folder + 'images_all_tic/', f"{common_group}.png"))
    plt.close(fig)


/tmp/ipykernel_348453/1726770718.py:68: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.96])  # leave space on right for cbar
/tmp/ipykernel_348453/1726770718.py:68: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.96])  # leave space on right for cbar
/tmp/ipykernel_348453/1726770718.py:68: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.96])  # leave space on right for cbar
/tmp/ipykernel_348453/1726770718.py:68: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.96])  # leave space on right for cbar
/tmp/ipykernel_348453/1726770718.py:68: UserWarning: This figure includes Axes that 